# 01 · Descarga de datos (S4)

Baja, para las 9 manchas de la muestra, los cuatro productos HMI a **720 s**:

| producto | serie JSOC | forma |
|---|---|---|
| continuo | `hmi.Ic_720s` | cutout con seguimiento (`im_patch`), malla HPC |
| magnetograma | `hmi.M_720s` | cutout con seguimiento, malla HPC |
| dopplergrama | `hmi.V_720s` | cutout con seguimiento, malla HPC |
| vector SHARP | `hmi.sharp_cea_720s` | HARP completo, malla CEA (Br, Bp, Bt, errores, bitmap, conf_disambig) |

**Ventanas de tiempo**
- *Löptien* (validación): el día completo de la fecha del barrido de Hinode. Es la misma ventana que `FIT_WINDOWS` del proyecto de oscilaciones, porque no hay registro de la hora exacta del barrido.
- *Catálogo propio* (predicción): las 48 h en que la mancha está más centrada en el disco, calculadas con las posiciones SRS de NOAA.

**Detalles heredados del proyecto `sun_spots_pulsation`**
- La caja del cutout la da `locate_ar_window`: es la caja HEK del grupo de manchas de NOAA SWPC.
- 11536 usa la caja más grande (`BOX_OVERRIDE`).
- 11117 baja las series de 45 s submuestreadas a 720 s, porque las de 720 s dan error 500 en JSOC.
- Las descargas se reanudan: correr de nuevo el notebook solo baja lo que falta.


In [1]:
import pathlib, sys, warnings
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)

from src import config, download
print('JSOC email:', config.NOTIFY_EMAIL)

JSOC email: thomas.quamtum@gmail.com


## Parámetros

- `REUSE_PREVIOUS`: copia los frames que el proyecto de oscilaciones ya bajó para las manchas de Löptien (misma caja, mismas series). Así no se vuelven a pedir unos 4 GB a JSOC.
- `DRY_RUN_HOURS`: si no es `None`, recorta todas las ventanas a sus primeras N horas. Sirve para probar el flujo completo bajando poco.
- `ONLY`: lista de `raw_id` para bajar solo esas carpetas (por ejemplo `['C12670']`); `None` baja todas.

In [2]:
REUSE_PREVIOUS = True
DRY_RUN_HOURS = None   # p. ej. 1 para una prueba rápida
ONLY = None

## 1. Ventanas de tiempo

Para el catálogo propio, `most_centered_window` hace estos pasos:
1. Ajusta una recta longitud(t) a las posiciones SRS diarias (≈13.2°/día).
2. Busca el instante del paso por el meridiano central (lon = 0).
3. Toma ±24 h alrededor de ese instante.

A latitud fija, μ es máximo en lon = 0 y simétrico alrededor, así que esa ventana es la de μ medio máximo. `mu_mean` y `mu_edge` son aproximados (sin B0).

In [3]:
rows = []
for s in config.LOPTIEN_SPOTS:
    rows.append(dict(id=s['id'], noaa=s['noaa'], raw_id=s['raw_id'], harpnum=s['harpnum'], kind='loptien',
                     t_start=s['t_start'], t_end=s['t_end']))
for s in config.CATALOG_SPOTS:
    w = download.most_centered_window(config.CATALOG_DIR / f"srs_{s['noaa']}.csv", hours=config.WINDOW_HOURS)
    rows.append(dict(id=s['id'], noaa=s['noaa'], raw_id=s['raw_id'], harpnum=s['harpnum'], kind='catalog',
                     t_start=w['t_start'], t_end=w['t_end'], t_cm=w['t_cm'],
                     mu_mean=w['mu_mean'], mu_edge=w['mu_edge'], rate_deg_day=w['rate_deg_per_day']))
windows = pd.DataFrame(rows)
for c in ('t_start', 't_end', 't_cm'):
    windows[c] = pd.to_datetime(windows[c])
if DRY_RUN_HOURS is not None:
    windows['t_end'] = windows['t_start'] + pd.Timedelta(hours=DRY_RUN_HOURS) - pd.Timedelta(seconds=1)
windows.to_csv(config.RAW_DIR / 'windows.csv', index=False)
windows

,id,noaa,raw_id,harpnum,kind,t_start,t_end,t_cm,mu_mean,mu_edge,rate_deg_day
0,L11106_20100916,11106,L11106,175,loptien,2010-09-16 00:00:00,2010-09-16 23:59:59,NaT,NaN,NaN,NaN
1,L11117_20101027,11117,L11117,226,loptien,2010-10-27 00:00:00,2010-10-27 23:59:59,NaT,NaN,NaN,NaN
2,L11117_20101028,11117,L11117,226,loptien,2010-10-28 00:00:00,2010-10-28 23:59:59,NaT,NaN,NaN,NaN
3,L11363_20111206,11363,L11363,1124,loptien,2011-12-06 00:00:00,2011-12-06 23:59:59,NaT,NaN,NaN,NaN
4,L11536_20120731,11536,L11536,1879,loptien,2012-07-31 00:00:00,2012-07-31 23:59:59,NaT,NaN,NaN,NaN
5,C12526,12526,C12526,6454,catalog,2016-03-29 08:24:00,2016-03-31 08:23:59,2016-03-30 08:24:00,0.987716,0.969924,13.363636
6,C12574,12574,C12574,6699,catalog,2016-08-12 16:12:00,2016-08-14 16:11:59,2016-08-13 16:12:00,0.987828,0.970930,13.023810
7,C12600,12600,C12600,6798,catalog,2016-10-11 11:00:00,2016-10-13 10:59:59,2016-10-12 11:00:00,0.971555,0.954038,13.369697
8,C12664,12664,C12664,7058,catalog,2017-06-25 13:12:00,2017-06-27 13:11:59,2017-06-26 13:12:00,0.944011,0.927284,13.254545
9,C12670,12670,C12670,7100,catalog,2017-08-06 10:12:00,2017-08-08 10:11:59,2017-08-07 10:12:00,0.986428,0.969411,13.078788


Comprobación del HARPNUM. Un HARP puede agrupar varias regiones: 1879 = 11529–11536 y 6699 = 12573–12577. Por eso la mancha se ubica con la caja SWPC y no con el centro del HARP.

In [4]:
for s in config.all_spots():
    t = pd.Timestamp(windows.set_index('id').loc[s['id'], 't_start']) + pd.Timedelta(hours=1)
    hit = download.find_harpnum(s['noaa'], t)
    ok = s['harpnum'] in hit['HARPNUM'].tolist()
    print(f"{s['id']:16s} HARP config {s['harpnum']:5d}  JSOC {hit['HARPNUM'].tolist()}  NOAA_ARS {hit['NOAA_ARS'].tolist()}  {'ok' if ok else 'REVISAR'}")

L11106_20100916  HARP config   175  JSOC [175]  NOAA_ARS ['11106']  ok


L11117_20101027  HARP config   226  JSOC [226]  NOAA_ARS ['11117']  ok


L11117_20101028  HARP config   226  JSOC [226]  NOAA_ARS ['11117']  ok


L11363_20111206  HARP config  1124  JSOC [1124]  NOAA_ARS ['11363,11373']  ok


L11536_20120731  HARP config  1879  JSOC [1879]  NOAA_ARS ['11529,11530,11532,11533,11536']  ok


C12526           HARP config  6454  JSOC [6454]  NOAA_ARS ['12526']  ok


C12574           HARP config  6699  JSOC [6699]  NOAA_ARS ['12573,12574,12575,12577']  ok


C12600           HARP config  6798  JSOC [6798]  NOAA_ARS ['12600']  ok


C12664           HARP config  7058  JSOC [7058]  NOAA_ARS ['12664']  ok


C12670           HARP config  7100  JSOC [7100]  NOAA_ARS ['12670']  ok


## 2. Una descarga por carpeta `raw_id`

Las dos fechas de 11117 comparten la carpeta `L11117`: su ventana de descarga es la unión, 27 y 28 de octubre.

In [5]:
groups = (windows.groupby('raw_id')
          .agg(noaa=('noaa', 'first'), harpnum=('harpnum', 'first'), kind=('kind', 'first'),
               t_start=('t_start', 'min'), t_end=('t_end', 'max')))
if ONLY is not None:
    groups = groups.loc[ONLY]
groups

,noaa,harpnum,kind,t_start,t_end
raw_id,,,,,
C12526,12526,6454,catalog,2016-03-29 08:24:00,2016-03-31 08:23:59
C12574,12574,6699,catalog,2016-08-12 16:12:00,2016-08-14 16:11:59
C12600,12600,6798,catalog,2016-10-11 11:00:00,2016-10-13 10:59:59
C12664,12664,7058,catalog,2017-06-25 13:12:00,2017-06-27 13:11:59
C12670,12670,7100,catalog,2017-08-06 10:12:00,2017-08-08 10:11:59
L11106,11106,175,loptien,2010-09-16 00:00:00,2010-09-16 23:59:59
L11117,11117,226,loptien,2010-10-27 00:00:00,2010-10-28 23:59:59
L11363,11363,1124,loptien,2011-12-06 00:00:00,2011-12-06 23:59:59
L11536,11536,1879,loptien,2012-07-31 00:00:00,2012-07-31 23:59:59


## 3. Reutilizar frames del proyecto de oscilaciones (solo Löptien)

In [6]:
if REUSE_PREVIOUS:
    for raw_id, g in groups.iterrows():
        prev = config.PREVIOUS_RAW_DIRS.get(raw_id)
        if prev is None:
            continue
        src = config.PREVIOUS_PROJECT_RAW / prev / 'region_01'
        for series in config.series_for(g['noaa']):
            meta = download.SERIES_META[series]
            n = download.reuse_previous_frames(src, config.RAW_DIR / raw_id / meta['label'],
                                               meta['glob'], g['t_start'], g['t_end'])
            print(f'{raw_id} {series}: {n} archivos copiados de {prev}')

L11106 hmi.Ic_720s: 0 archivos copiados de NOAA_11106_2010-09-16
L11106 hmi.M_720s: 0 archivos copiados de NOAA_11106_2010-09-16
L11106 hmi.V_720s: 0 archivos copiados de NOAA_11106_2010-09-16
L11117 hmi.Ic_45s: 0 archivos copiados de NOAA_11117_2010-10-27
L11117 hmi.M_45s: 0 archivos copiados de NOAA_11117_2010-10-27
L11117 hmi.V_45s: 0 archivos copiados de NOAA_11117_2010-10-27
L11363 hmi.Ic_720s: 0 archivos copiados de NOAA_11363_2011-12-06
L11363 hmi.M_720s: 0 archivos copiados de NOAA_11363_2011-12-06
L11363 hmi.V_720s: 0 archivos copiados de NOAA_11363_2011-12-06
L11536 hmi.Ic_720s: 0 archivos copiados de NOAA_11536_2012-07-31
L11536 hmi.M_720s: 0 archivos copiados de NOAA_11536_2012-07-31
L11536 hmi.V_720s: 0 archivos copiados de NOAA_11536_2012-07-31


## 4. Cajas HEK y descarga de cutouts

La caja de Löptien se calcula con el mismo ancla y los mismos `n_days = 4` del proyecto original, para que coincida con los frames reutilizados. En el catálogo propio, el ancla es el primer día de la ventana.

`download_cutouts` devuelve uno de cuatro estados:
- `skipped`: ya estaba en disco.
- `ok`
- `partial`: faltan archivos tras el reintento. Hay que volver a correr la celda.
- `failed`

In [7]:
status = []
boxes = {}
for raw_id, g in groups.iterrows():
    noaa = g['noaa']
    if g['kind'] == 'loptien':
        anchor, n_days = config.PREVIOUS_RAW_DIRS[raw_id].split('_')[-1], 4
    else:
        anchor = f"{g['t_start']:%Y-%m-%d}"
        n_days = (g['t_end'].normalize() - g['t_start'].normalize()).days + 1
    bl, tr, _, _ = download.locate_ar_window(noaa, anchor, n_days=n_days, **config.BOX_OVERRIDE.get(noaa, {}))
    boxes[raw_id] = (bl, tr)
    for series in config.series_for(noaa):
        meta = download.SERIES_META[series]
        r = download.download_cutouts(bl, tr, config.RAW_DIR / raw_id / meta['label'],
                                      str(g['t_start']), str(g['t_end']), config.NOTIFY_EMAIL,
                                      series, sample=config.sample_for(noaa))
        status.append(dict(raw_id=raw_id, product=series, **{k: v for k, v in r.items() if k != 'failed_urls'}))

NOAA 12526: box 180 arcsec square, centered -297 arcsec,40 arcsec — from 3 NOAA SWPC Observer row(s), median half-width 25 arcsec
[hmi.Ic_720s] ya cubre el rango hasta 2016-03-31 08:24:00 — se salta
[hmi.M_720s] ya cubre el rango hasta 2016-03-31 08:24:00 — se salta
[hmi.V_720s] ya cubre el rango hasta 2016-03-31 08:24:00 — se salta


NOAA 12574: box 180 arcsec square, centered -340 arcsec,-34 arcsec — from 3 NOAA SWPC Observer row(s), median half-width 41 arcsec
[hmi.Ic_720s] ya cubre el rango hasta 2016-08-14 16:12:00 — se salta
[hmi.M_720s] ya cubre el rango hasta 2016-08-14 16:12:00 — se salta
[hmi.V_720s] ya cubre el rango hasta 2016-08-14 16:12:00 — se salta


NOAA 12600: box 180 arcsec square, centered -308 arcsec,87 arcsec — from 3 NOAA SWPC Observer row(s), median half-width 17 arcsec
[hmi.Ic_720s] ya cubre el rango hasta 2016-10-13 11:00:00 — se salta
[hmi.M_720s] ya cubre el rango hasta 2016-10-13 11:00:00 — se salta
[hmi.V_720s] ya cubre el rango hasta 2016-10-13 11:00:00 — se salta


NOAA 12664: box 180 arcsec square, centered -323 arcsec,260 arcsec — from 3 NOAA SWPC Observer row(s), median half-width 55 arcsec
[hmi.Ic_720s] ya cubre el rango hasta 2017-06-27 13:12:00 — se salta
[hmi.M_720s] ya cubre el rango hasta 2017-06-27 13:12:00 — se salta
[hmi.V_720s] ya cubre el rango hasta 2017-06-27 13:12:00 — se salta


NOAA 12670: box 180 arcsec square, centered -323 arcsec,-193 arcsec — from 3 NOAA SWPC Observer row(s), median half-width 58 arcsec
[hmi.Ic_720s] ya cubre el rango hasta 2017-08-08 10:12:00 — se salta
[hmi.M_720s] ya cubre el rango hasta 2017-08-08 10:12:00 — se salta
[hmi.V_720s] ya cubre el rango hasta 2017-08-08 10:00:00 — se salta


NOAA 11106: box 329 arcsec square, centered -278 arcsec,-432 arcsec — from 4 NOAA SWPC Observer row(s), median half-width 134 arcsec
[hmi.Ic_720s] ya cubre el rango hasta 2010-09-16 23:48:00 — se salta
[hmi.M_720s] ya cubre el rango hasta 2010-09-16 23:48:00 — se salta
[hmi.V_720s] ya cubre el rango hasta 2010-09-16 23:48:00 — se salta


NOAA 11117: box 202 arcsec square, centered 234 arcsec,272 arcsec — from 4 NOAA SWPC Observer row(s), median half-width 71 arcsec
[hmi.Ic_45s] ya cubre el rango hasta 2010-10-28 23:48:45 — se salta
[hmi.M_45s] ya cubre el rango hasta 2010-10-28 23:48:45 — se salta
[hmi.V_45s] ya cubre el rango hasta 2010-10-28 23:48:45 — se salta


NOAA 11363: box 220 arcsec square, centered 143 arcsec,-355 arcsec — from 4 NOAA SWPC Observer row(s), median half-width 80 arcsec
[hmi.Ic_720s] ya cubre el rango hasta 2011-12-06 23:48:00 — se salta
[hmi.M_720s] ya cubre el rango hasta 2011-12-06 23:48:00 — se salta
[hmi.V_720s] ya cubre el rango hasta 2011-12-06 23:48:00 — se salta


NOAA 11536: box 315 arcsec square, centered -383 arcsec,-447 arcsec — from 4 NOAA SWPC Observer row(s), max half-width 98 arcsec
[hmi.Ic_720s] ya cubre el rango hasta 2012-07-31 23:48:00 — se salta
[hmi.M_720s] ya cubre el rango hasta 2012-07-31 23:48:00 — se salta
[hmi.V_720s] ya cubre el rango hasta 2012-07-31 23:48:00 — se salta


## 5. Vector SHARP (`hmi.sharp_cea_720s`)

Se exporta con drms (`protocol='fits'`, así cada segmento trae todas sus keywords y su WCS CEA). JSOC permite **una** exportación pendiente por correo; `download_sharp` espera si hay otra en cola. El HARP 1879 (11536) es enorme, ~21 MB por registro con los 8 segmentos; en 02 se recorta a la zona del cutout.

In [8]:
for raw_id, g in groups.iterrows():
    r = download.download_sharp(g['harpnum'], g['t_start'], g['t_end'], config.RAW_DIR / raw_id / 'sharp',
                                config.NOTIFY_EMAIL, series=config.SHARP_SERIES, segments=config.SHARP_SEGMENTS)
    status.append(dict(raw_id=raw_id, product=config.SHARP_SERIES, **r))

2026-11-16 01:39:07 - drms - INFO: Export request pending. [id=JSOC_20261116_001194, status=2]


2026-11-16 01:39:07 - drms - INFO: Waiting for 10 seconds...


2026-11-16 01:39:18 - drms - INFO: Export request pending. [id=JSOC_20261116_001194, status=1]


2026-11-16 01:39:18 - drms - INFO: Waiting for 10 seconds...


2026-11-16 01:39:29 - drms - INFO: Export request pending. [id=JSOC_20261116_001194, status=1]


2026-11-16 01:39:29 - drms - INFO: Waiting for 10 seconds...


... (39,775 outputs left out)

[SHARP 175] FALLÓ — TimeoutError('The read operation timed out')


[SHARP 226] FALLÓ — TimeoutError('timed out')


[SHARP 1124] FALLÓ — TimeoutError('timed out')


[SHARP 1879] FALLÓ — TimeoutError('timed out')


## 6. Estado final

Número de frames por producto en cada carpeta. Una ventana de 24 h a 720 s tiene 120 slots y una de 48 h tiene 240. Los faltantes que reporta JSOC se ven aquí y luego en 02, que solo usa los tiempos donde están los cuatro productos.

In [9]:
status_df = pd.DataFrame(status)
display(status_df)

from src.download import PRODUCT_GLOBS
count = {}
for raw_id in groups.index:
    d = config.RAW_DIR / raw_id
    count[raw_id] = {p: len(list((d / p).glob(gl))) for p, gl in PRODUCT_GLOBS.items()}
    count[raw_id]['sharp_Br'] = len(list((d / 'sharp').glob('*.Br.fits')))
pd.DataFrame(count).T

,raw_id,product,status,n_files,n_records,error
0,C12526,hmi.Ic_720s,skipped,240,NaN,NaN
1,C12526,hmi.M_720s,skipped,240,NaN,NaN
2,C12526,hmi.V_720s,skipped,240,NaN,NaN
3,C12574,hmi.Ic_720s,skipped,241,NaN,NaN
4,C12574,hmi.M_720s,skipped,241,NaN,NaN
5,C12574,hmi.V_720s,skipped,241,NaN,NaN
6,C12600,hmi.Ic_720s,skipped,240,NaN,NaN
7,C12600,hmi.M_720s,skipped,240,NaN,NaN
8,C12600,hmi.V_720s,skipped,240,NaN,NaN
9,C12664,hmi.Ic_720s,skipped,241,NaN,NaN


,continuum,magnetogram,dopplergram,sharp_Br
C12526,240,240,240,239
C12574,241,241,241,238
C12600,240,240,240,240
C12664,241,241,241,241
C12670,241,241,240,241
L11106,114,114,114,45
L11117,234,234,234,0
L11363,120,119,120,0
L11536,120,120,120,6
